<a href="https://colab.research.google.com/github/leejunho12316/HonGongMachine/blob/main/3_%EC%97%AC%EB%9F%AC_%EB%AC%B8%EC%84%9C%EB%A5%BC_%EC%9D%B8%EC%9A%A9%ED%95%B4%EC%95%BC_%ED%95%98%EB%8A%94_%EC%A7%88%EB%AC%B8_%EB%A7%8C%EB%93%A4%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#환경설정

In [2]:
!pip install datasets openai

#데이터셋 불러오기

In [3]:
from datasets import load_dataset

# 1. Hugging Face에서 데이터셋 불러오기
dataset = load_dataset("iamjoon/klue-mrc-bge-m3")

# 2. train split을 데이터프레임으로 변환
df = dataset['train'].to_pandas()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/640 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/77.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10434 [00:00<?, ? examples/s]

In [4]:
df = df[:10]

#여러 문서를 인용하는 질문 만들기

검색결과 (pos _ 4neg) - shuffle 안함

In [5]:
samples = []
for context, negative_samples in zip(df['context'].to_list(), df['negative_samples'].to_list()):
  samples.append([context] + list(negative_samples))

프롬프트 제작

In [6]:
user_prompts = []
for sample in samples:
  user_prompt = ''
  for idx, s in enumerate(sample):
    user_prompt = user_prompt + '문서' + str(idx + 1) + ':' + s + '\n'
    user_prompt = user_prompt + '---\n'
  user_prompts.append(user_prompt)

In [7]:
system_prompt = """당신은 주어진 문서로부터 2~3개 이상의 문서를 인용해야만 답변이 가능한 질문 5개를 제안해야 합니다.

### 구체적인 지시사항
1. 문서를 5개 드리겠습니다. 2~3개 이상의 문서를 인용해야만 답변 가능한 질문 5개를 제안하세요.
2. 질문을 적고 '어떤 문서들'을 '왜' 인용해야 답변 가능한 질문인지 그 근거를 적으시기 바랍니다.
3. 질문 다음에 어떠한 이유로 두 개 문서 인용이 가능할 것 같은지 근거를 적습니다. 해당 문서들이 질문에 부합하여 인용 가능하다는 근거가 있어야 합니다.
4. 질문은 너무 지엽적이고 너무 긴 질문은 지양합니다. 이는 매우 중요합니다. 질문은 매우 짧게 작성하십시오.

하지 말아야 할 질문과 해야할 질문에 대해서 예를 들어보겠습니다.
###
질문: 한국전력 본사 건물의 매각 여부와 관련하여 다양한 입장이 존재하는데, 어떤 문서들이 다양한 입장을 다루고 있으며, 그 입장은 어떤 내용인가요?
=> 위 질문은 짧게 작성하라는 요구 사항을 무시한 경우입니다. 이렇게 작성하지 마십시오.

질문: 에너지 기본권의 개념과 관련 법안에 대해 설명해줘
=> 위와 같이 작성하십시오.
###

5. 결정:에서는 근거를 읽어보고 결과적으로 두 개 이상의 문서를 인용하고 있는지 살펴보고 => 다음에 True or False를 적습니다. 만약 두 개 이상의 문서 인용을 근거로 들고 있지 않다면 False 처리 하십시오.
6. 결정:을 작성할 때에는 근거를 검토하는 의견을 적은 뒤에 다음에 => True or False를 적습니다. True or False 뒤에는 아무 것도 적지 마십시오.
True or False의 예시는 다음과 같습니다.
###
질문2: 최근 SF 영화들의 주제와 특징을 알려줘
근거2: 최근 SF 영화들의 주제와 특징에 대해서는 문서3에서 다루는 다양한 SF 영화들의 주제와 특징을 통해 추측 가능함. 예시로 '스타트렉 다크니스', '애프터 어스', '맨 오브 스틸' 등의 영화가 있음.
결정2: 근거2를 읽어보면 문서3을 통해 답변 가능하다고 하고 있으며 이는 단일 문서이므로 지시사항을 위반한다. => False

질문3: SF 영화들이 주제로 다루는 인간과 기술의 관계는 어떻게 변화해왔지?
근거3: SF 영화들이 주제로 다루는 인간과 기술의 관계는 문서3에서 다양한 SF 영화들이 보여주는 인간과 기술의 관계를 다룸, 예를 들어 '스타트렉다크니스', '애프터 어스', '엘리시움' 등에서 어떻게 기술이 인간 생활에 영향을 미치는지 설명함.
결정3: 근거2를 읽어보면 문서3을 통해 답변 가능하다고 하고 있으며 이는 단일 문서이므로 지시사항을 위반한다. => False

질문4: 북한과 관련된 소재를 다루는 소설과 영화에서 나타나는 주제적 차이는 무엇일까?
근거4: 북한과 관련된 소재를 다루는 소설과 영화에서 나타나는 주제적 차이는 문서1과 문서3 을 통해 답변 가능하다. 문서1의 전성태 소설집에서 새터민과 북한 신문 등의 소재를 다루는 복수의 소설들, 문서3에서는 '백악관 최후의 날'과 같은 북한 관련 영화 주제를 다룸.
결정4: 문서1과 문서3에서 각각 소설과 영화에서 북한 관련 소재를 다루는 방법과 주제적 차이를 비교할 수 있으며 두 개 이상의 문서를 인용할 수 있다고 한다. => True
###
7. 형식은 다음과 같습니다.

답변:
질문1: 질문을 적으세요
근거1: 두 개 이상의 문서를 인용할 수 있다는 근거를 적으세요.
결정1: 근거1에서 1개 문서만 언급되었다거나 두 개 이상의 문서를 인용할 수 있다는 근거가 빈약했다면 최종적으로 False를 적고 근거에서 두 개 이상의 문서 인용할 수 있다는 근거가 나왔다면 True로 적으세요. => 다음에 적습니다.
...중략...
질문5: 질문을 적으세요
근거5: 두 개 이상의 문서를 인용할 수 있다는 근거를 적으세요
결정5: 근거5에서 1개 문서만 언급되었다거나 두 개 이상의 문서를 인용할 수 있다는 근거가 빈약했다면 최종적으로 False를 적고 근거에서 두 개 이상의 문서 인용할 수 있다는 근거가 나왔다면 True로 적으세요. => 다음에 적습니다."""

In [52]:
import openai
client = openai.OpenAI(api_key="")

In [13]:
from tqdm import tqdm

result_lst = []
for user_prompt in tqdm(user_prompts):
  response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt + '답변:'}
      ]
  )
  result_lst.append(response.choices[0].message.content)

100%|██████████| 10/10 [01:38<00:00,  9.81s/it]


In [32]:
import re
import random

#1. 근거 빈약한 질문 삭제하기
def remove_false_decisions(text):
    pattern = r'(질문\d+:.+?\n근거\d+:.+?\n결정\d+:.+?(?:True|False))\n*'

    sets = re.findall(pattern, text, re.DOTALL)
    filtered_sets = [s for s in sets if "=> False" not in s]

    result = "\n\n".join(filtered_sets)

    return result

#2. 질문-근거-결정 중 질문만 파싱
def extract_questions(text):
    pattern = r'질문\d+: (.+?)(?=\n근거\d+:|$)'

    questions = re.findall(pattern, text, re.DOTALL)

    questions = [q.strip() for q in questions]

    return questions

#3. 여러 질문중 하나만 선택
def select_random_question(questions_list):
    if not questions_list:
        return None  # 리스트가 비어있을 경우 None 반환
    return random.choice(questions_list)

In [33]:
one_questions = []
for five_questions in result_lst:
  true_questions = remove_false_decisions(five_questions)
  questions_list = extract_questions(true_questions)
  random_question = select_random_question(questions_list)
  one_questions.append(random_question)

In [35]:
df['multi_document_question'] = one_questions
df.head()

,title,news_category,source,context,question,question_type,is_impossible,answer_text,answer_start,negative_samples,multi_document_question
0,제주도 장마 시작 … 중부는 이달 말부터,종합,hankyung,올여름 장마가 17일 제주도에서 시작됐다. 서울 등 중부지방은 예년보다 사나흘 정도...,북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?,1,False,한 달가량,478,[궤도물리학은 계절의 지속 기간이 지구의 궤도가 지점과 분점 사이의 공간을 휩쓸고 ...,지구의 궤도와 기후 변화의 상관관계에 대해 설명해줘
1,"부산정보산업진흥원, 과기부 지역SW서비스사업화 지원사업 4개 과제 선정",경제,acrofan,부산시와 (재)부산정보산업진흥원(원장 이인숙)이 ‘2020~2021년 지역SW서비스...,지능형 생산자동화 기반기술을 개발중인 스타트업은?,2,False,삼보테크놀로지,1422,"[IIoT(Industry Internet of Thing), 산업현장 스마트 팩토...",국내 사물인터넷(IoT) 연관 산업의 트렌드와 관련 기업의 협력 방식은?
2,나루세 요시히사,None,wikipedia,시범 경기에서는 16이닝을 던져 15실점을 기록하는 등 성적이 좋지 않았지만 본인으...,개막전에서 3안타 2실점을 기록해서 패한 선수는?,2,False,와쿠이 히데아키,107,[1960년에 주니치에 입단하여 같은 해 1960년 5월 7일 다이요 웨일스전에서 ...,일본 프로 야구 개막전에서의 투수진의 경기 결과와 관련된 기록은?
3,편의점 휩쓴 ‘맛집 라면’,생활경제,hankyung,유명 맛집 이름을 달고 나온 편의점 자체상표(PB) 라면이 인기를 끌고 있다. ‘검...,컵라면 매출에서 불닭볶음면을 이긴 상품은?,2,False,‘교동반점 짬뽕’,408,[‘짜파구리’로 시작된 ‘나만의 레시피’ 열풍이 올해 식품시장을 주도한 것으로 나타...,편의점 자체상표(PB) 제품의 라면과 팝콘의 인기 요인은 무엇인가요?
4,사모펀드 ACP 아난드 프라카시 전무 ...“아시아 친환경 투자 황금기 온다”,기획,hankyung,“앞으로 5년 안에 아시아 친환경·신재생에너지 투자의 황금기가 도래할 것입니다.”기...,정부에게 환경과 관련해서 우선적으로 원조 받고 있는 곳은?,1,False,환경오염이 심한 지역,661,[포브스는 최근 “석탄은 기원전 315년 그리스 문헌에 대장간에서 원료로 쓰던 기록...,전기차 보급 확대를 위한 정부의 방안과 그에 따른 환경적 영향은 무엇인가?


#답변 생성

In [ ]:
새로 생성한 질문 - pos - 4neg
->
새로 생성한 질문 - 검색 결과 - 답변 - 참고문

In [39]:
user_prompt_for_answer = []
for multi_document_question, user_prompt in zip(one_questions, user_prompts):
  user_prompt_for_answer.append('질문:' + multi_document_question + '\n' + user_prompt + '답변:')

In [40]:
system_prompt = """당신은 검색된 문서부터 질문의 답변을 작성하는 언어 모델입니다.

### 지시사항
1. 질문으로부터 주어진 다수의 문서에서 답을 찾아 작성하세요.
2. 검색된 문서에 질문의 답이 없는 경우에는 질문에 대한 내용을 언급하면서 답을 찾을 수 없다고 작성하세요.
3. 답변에서 참고자료의 내용을 인용한 경우, 답변 맨 끝에 인용한 문서 id를 추가하세요. 답변에 내용을 인용한 경우에만 작성해야 합니다. 인용한 게 없다면 쓰지마세요.
4. 문서 id 값은 검색 결과에서 가져온 것이며 두 개의 대괄호로 묶어야 하며 ref라는 키워드와 함께 작성해야 합니다. 예를 들어 특정 문단이 3번, 5번 문서에서 인용했다면 다음과 같이 작성하십시오.
예시) ~~~해서 했습니다. [[ref3]], [[ref5]]
5. 주요 문장 또는 문단 뒤에 ref문서번호를 덧붙이는 것은 매우 중요합니다. 이는 반드시 지켜져야 합니다.
6. 문단 중간 중간에 인용이 들어가더라도 답변은 하나의 이어지는 평문으로 작성하십시오.
7. 주어진 문서는 검색 결과입니다. 따라서 질문과 연관없는 문서가 숨어있을 수 있으니 이를 인용하지 않도록 주의하십시오.
8. 검색 결과에 없는 내용은 작성하지 마십시오. 가장 중요한 지시사항입니다. 따라서 인용 없이는 그 어떠한 설명도 제공할 수 없습니다.
9. 답변은 최대한 풍부하게 작성해주시기 바랍니다.
"""

In [41]:
# 5개로 테스트
result_lst_for_answer = []
for user_prompt in tqdm(user_prompt_for_answer):
  response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt}
      ],
    temperature=0
  )
  result_lst_for_answer.append(response.choices[0].message.content)

100%|██████████| 10/10 [00:58<00:00,  5.81s/it]


In [42]:
df['multi_document_answer'] = result_lst_for_answer

In [43]:
# 인용 문서 번호 추출
def extract_ref_numbers(text):
    # 정규표현식을 사용하여 [[ref숫자]] 패턴 찾기
    pattern = r'\[\[ref(\d+)\]\]'

    # 모든 매치 찾기
    matches = re.findall(pattern, text)

    # 문자열을 정수로 변환하고 중복 제거 후 정렬
    unique_numbers = sorted(set(map(int, matches)))

    return unique_numbers

In [45]:
extracted_ref_numbers = []
for sample in result_lst_for_answer:
  extracted_ref_numbers.append(extract_ref_numbers(sample))

df['extracted_ref_numbers'] = extracted_ref_numbers

In [48]:
samples = []
for context, negative_samples in zip(df['context'].to_list(), df['negative_samples'].to_list()):
  samples.append([context] + list(negative_samples))

df['search_result'] = samples

In [51]:
df[['multi_document_question','search_result', 'multi_document_answer','extracted_ref_numbers']]

,multi_document_question,search_result,multi_document_answer,extracted_ref_numbers
0,지구의 궤도와 기후 변화의 상관관계에 대해 설명해줘,[올여름 장마가 17일 제주도에서 시작됐다. 서울 등 중부지방은 예년보다 사나흘 정...,지구의 궤도와 기후 변화의 상관관계는 주로 지구의 궤도 변화가 기후에 미치는 영향을...,[2]
1,국내 사물인터넷(IoT) 연관 산업의 트렌드와 관련 기업의 협력 방식은?,[부산시와 (재)부산정보산업진흥원(원장 이인숙)이 ‘2020~2021년 지역SW서비...,국내 사물인터넷(IoT) 연관 산업의 트렌드와 관련 기업의 협력 방식에 대해 다음과...,"[2, 4, 5]"
2,일본 프로 야구 개막전에서의 투수진의 경기 결과와 관련된 기록은?,[시범 경기에서는 16이닝을 던져 15실점을 기록하는 등 성적이 좋지 않았지만 본인...,일본 프로 야구 개막전에서의 투수진의 경기 결과와 관련된 기록은 문서1에서 찾을 수...,[1]
3,편의점 자체상표(PB) 제품의 라면과 팝콘의 인기 요인은 무엇인가요?,[유명 맛집 이름을 달고 나온 편의점 자체상표(PB) 라면이 인기를 끌고 있다. ‘...,편의점 자체상표(PB) 제품의 라면과 팝콘의 인기는 여러 요인에 기인합니다.\n\n...,"[1, 4]"
4,전기차 보급 확대를 위한 정부의 방안과 그에 따른 환경적 영향은 무엇인가?,[“앞으로 5년 안에 아시아 친환경·신재생에너지 투자의 황금기가 도래할 것입니다.”...,전기차 보급 확대를 위한 정부의 방안으로는 주로 일반 소비자와 관공서에 판매하던 방...,[3]
5,팬데믹 상황에서 비즈니스 모델에 변화를 준 사례는 어떤 것이 있나요?,[아랍 에미리트의 국영항공사 에티하드항공이 지난 10월 21일 열린 ’2020비즈니...,팬데믹 상황에서 비즈니스 모델에 변화를 준 사례로는 에티하드항공과 위워크가 있습니다...,"[1, 2]"
6,역사적 인물의 과학과 기술에 대한 기여도에 대해 설명해줘,[방송일시(인터넷 LIVE) : 2020년 12월 10일 (목) 낮 15:00 – ...,역사적 인물 중 존 디는 과학과 기술 분야에 여러 기여를 한 인물로 알려져 있습니다...,[5]
7,코로나19 시대에 예술 공연과 관련된 관객의 경험은 어떻게 변화했나요?,[(재)정동극장(대표이사 김희철)은 정부가 23일 코로나 19 위기 경보를 ‘심각’...,코로나19 시대에 예술 공연과 관련된 관객의 경험은 여러 가지 방식으로 변화했습니다...,"[1, 2, 4]"
8,"삼성, 포스코, SK의 경영 전략과 철학의 차이점은 무엇인가?",[“제트기가 음속을 돌파하려면 비행기의 모든 소재가 다 바뀌어야 한다. 재료공학부터...,"삼성, 포스코, SK의 경영 전략과 철학은 각기 다른 방향성과 중점을 가지고 있습니...","[1, 2, 5]"
9,다양한 e스포츠 대회에서 방송되는 경로에 대해 알아보자,[㈜넥슨(대표 이정헌)은 온라인 레이싱게임 ‘카트라이더’에서 진행하는 e스포츠 대회...,"다양한 e스포츠 대회는 여러 경로를 통해 방송됩니다. 예를 들어, '2020 카트라...","[1, 2, 3, 4, 5]"
